# Trends: vehicle stops and collisions by month

**A1.** How have vehicle stops and collisions changed over time, and do they move together month by month?

**Inputs:** `data/processed/vehicle_stops.csv`, `data/processed/collisions.csv`

**Outputs:** `figures/02_a1_monthly_trends.png`, `figures/02_a1_stops_vs_collisions.png`, `results/02_a1_monthly_counts.csv`, `results/02_a1_yearly_totals.csv`, `results/02_a1_correlation.csv`

In [1]:
# Shared loaders, the monthly counter, plot style, and project settings
import pandas as pd

from bpdtools.config import ANALYSIS_END
from bpdtools.paths import repo_path
from bpdtools.plots import set_plot_style
from bpdtools.traffic import load_collisions, load_stops
from bpdtools.trends import monthly_counts

set_plot_style()

# Figures and result tables go to fixed folders at the repository root
figures = repo_path("figures")
results = repo_path("results")
figures.mkdir(exist_ok=True)
results.mkdir(exist_ok=True)

# Stop Data starts in January 2024 and the analysis window ends in
# December 2025, so both sources cover the same 24 whole months
FIRST_MONTH = pd.Period("2024-01", freq="M")
LAST_MONTH = pd.Period(ANALYSIS_END, freq="M")

## Monthly counts

Each stop counts once by `LEA_Record_ID`. A collision without a date has no month and is left out.

In [2]:
# Count each vehicle stop once by its record id, in the month of its date
stops = load_stops()
stop_months = monthly_counts(stops, "StopDate", id_column="LEA_Record_ID")

# Collisions go back to 2007; monthly_counts skips the rows without a date
collisions = load_collisions()
collision_months = monthly_counts(collisions, "Date")

# Keep the 24 months that both sources cover
monthly = pd.DataFrame(
    {
        "vehicle_stops": stop_months.loc[FIRST_MONTH:LAST_MONTH],
        "collisions": collision_months.loc[FIRST_MONTH:LAST_MONTH],
    }
)
monthly.to_csv(results / "02_a1_monthly_counts.csv")

# Checks: the number of months, missing values, and undated collisions
checks = pd.Series(
    {
        "months": len(monthly),
        "missing values": int(monthly.isna().sum().sum()),
        "collisions without a date": int(collisions["Date"].isna().sum()),
    },
    name="value",
)
checks

months                        24
missing values                 0
collisions without a date    101
Name: value, dtype: int64